<a href="https://colab.research.google.com/github/wjdwogns2873-web/deep-learning-study/blob/main/05_Intel_Image_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!kaggle datasets download -d puneet6060/intel-image-classification
!unzip -q intel-image-classification.zip -d ./intel_images

In [ ]:
import os

base_dir = './intel_images'
print(os.listdir(base_dir))

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. 이미지 전처리 정의
# Intel 이미지 데이터셋은 기본적으로 150 150 크기입니다.
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor() # 이미지를 [0, 1] 값을 가진 텐서로 변환
])

# 2. ImageFolder를 이용해 데이터셋 로드
train_dir = './intel_images/seg_train/seg_train'
train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)

# 3. DataLoader 생성
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# 클래스 이름 확인
class_names = train_dataset.classes
print("클래스 종류:", class_names)

In [ ]:
print(len(train_dataset), len(train_loader))

In [ ]:
images, labels = next(iter(train_loader))

print(images.shape, labels.shape)

fig, axes = plt.subplots(1, 4, figsize=(12, 4))

for i in range(4):
    img = images[i]
    label = labels[i].item()

    # [3, 150, 150] -> [150, 150, 3]
    img_permuted = img.permute(1, 2, 0)

    axes[i].imshow(img_permuted)
    axes[i].set_title(class_names[label])
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(128 * 18 * 18, 512)
        self.fc2 = nn.Linear(512, 6)
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN()

In [ ]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 3

model.train()

for epoch in range(epochs):
    running_loss, epoch_loss = 0.0, 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        epoch_loss += loss.item()

        if (i+1) % 100 == 0:
            print(f"Epoch {epoch+1} | Batch {i+1} | Loss: {running_loss/100:.4f}")
            running_loss = 0.0

    print(f"==> Epoch {epoch+1} 완료! 평균 Loss: {epoch_loss/len(train_loader):.4f}")

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# 1. ImageNet 가중치가 포함된 ResNet18 모델 불러오기
# 최신 파이토치 버전에서는 weights 설정을 권장합니다.
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 2. ResNet의 마지막 레이어(fc) 구조 확인하기
in_features = resnet.fc.in_features
print(in_features)

# 3. 마지막 출력층을 우리의 6개의 클래스(풍경)에 맞게 새로 정의하기
# 이 한 줄로 전이 학습을 위한 모델 개조가 끝납니다!
resnet.fc = nn.Linear(in_features, 6)

resnet = resnet.to(device)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

# 개조된 resnet의 파라미터를 업데이트하도록 Adam 옵티마이저에 연결합니다.
optimizer_resnet = optim.Adam(resnet.parameters(), lr=0.0001)

In [ ]:
epochs = 3


for epoch in range(epochs):
    resnet.train()
    running_loss, epoch_loss = 0.0, 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer_resnet.zero_grad()
        outputs = resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_resnet.step()

        running_loss += loss.item()
        epoch_loss += loss.item()

        if (i+1) % 200 == 0:
            print(f"ResNet | Epoch {epoch+1} | Batch {i+1} | Loss: {running_loss/100:.4f}")
            running_loss = 0.0

    print(f"==> ResNet Epoch {epoch+1} 완료! 평균 Loss: {epoch_loss/len(train_loader):.4f}")

In [ ]:
# 1. 테스트 데이터셋 및 데이터로더 설정
test_dir = './intel_images/seg_test/seg_test'
test_dataset = datasets.ImageFolder(root=test_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

resnet.eval()

correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = resnet(images)

        # 가장 높은 확률(값)을 가진 인덱스(클래스) 추출
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy: {accuracy:.2f}")

In [ ]:
# 1. 훈련 데이터용 전처리 (데이터 증강 기법 추가!)
train_transform_augmented = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.RandomHorizontalFlip(p=0.5), # 50% 확률로 좌우 반전
    transforms.RandomRotation(degrees=15), # -15도 ~ +15도 무작위 회전
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # 밝기 및 대비 조절
    transforms.ToTensor()
])

# 2. 테스트 데이터용 전처리 (테스트 데이터는 증강을 하면 안 됩니다.)
test_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor()
])

# 3. 새로운 데이터셋 및 데이터로더 생성
train_dir = './intel_images/seg_train/seg_train'
test_dir = './intel_images/seg_test/seg_test'

augmented_train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform_augmented)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transform)

train_loader_aug = DataLoader(augmented_train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(len(augmented_train_dataset), len(train_loader_aug))

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
import torchvision.models as models
import torch.optim as optim
import torch.nn as nn

# 1. 완전히 새로운 ResNet18 인스턴스 생성 및 개조
model_aug = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
in_features = model_aug.fc.in_features
model_aug.fc = nn.Linear(in_features, 6)
model_aug = model_aug.to(device)

# 2. 손실함수 및 옵티마이저 재정의
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_aug.parameters(), lr=0.0001)

# 3. 데이터 증강을 적용한 학습 시작
epochs = 3

for epoch in range(epochs):
    model_aug.train()
    epoch_loss = 0.0
    for i, (images, labels) in enumerate(train_loader_aug):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model_aug(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Augmented ResNet | Epoch {epoch+1} 완료! 평균 Loss: {epoch_loss / len(train_loader_aug):.4f}")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

temp_test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

# images, labels = next(iter(test_loader))
images, labels = next(iter(temp_test_loader))

model_aug.eval()
with torch.no_grad():
    outputs = model_aug(images.to(device))
    _, preds = torch.max(outputs, 1)

images = images.cpu()
labels = labels.cpu().numpy()
preds = preds.cpu().numpy()

# 무작위로 6개의 인덱스 뽑기
random_indices = np.random.choice(len(images), 6, replace=False)

# 2행 3열의 캔버스 생성하여 시각화
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel() # 2D 배열을 1D 배열로 펼쳐서 반복문 쓰기 편하게 변환

for i, idx in enumerate(random_indices):
    img = images[idx].permute(1, 2, 0) # [C, H, W] -> [H, W, C]
    true_label = class_names[labels[idx]]
    pred_label = class_names[preds[idx]]

    color = 'blue' if true_label == pred_label else 'red'

    axes[i].imshow(img)
    axes[i].set_title(f"True: {true_label}\nPred: {pred_label}", color=color, fontsize=14)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 1. 모든 테스트 데이터에 대한 예측값과 정답 모으기
all_preds, all_labels = [], []

model_aug.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model_aug(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# 2. Confusion Matrix 계산
cm = confusion_matrix(all_labels, all_preds)

# 3. Seaborn을 이용해 히트맵(Heatmap) 시각화
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)

plt.xlabel('Predicted (Model)', fontsize=12)
plt.ylabel('True (Actual)', fontsize=12)
plt.title('Intel Image Classification - Confusion Matrix', fontsize=15)
plt.show()